In [ ]:
#se instala para poder separar las imagenes

In [10]:
#imports
import tensorflow as tf
import keras
import os
#librerias para el entrenamiento del modelo
from keras.models import Sequential, load_model
from keras.layers import Dense, Conv2D, Flatten, Dropout, MaxPooling2D, Input

#librerias para mostrar los cambios de la funcion loss
from keras.src.legacy.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import numpy as np
import splitfolders #para separar el dataset en dif carpetas mientras se entrena todo
import cv2

from keras import layers, models
from keras.applications.resnet50 import preprocess_input


In [12]:
input_folder = "dataset"
output_folder = "image-recognition"

In [13]:
split_ratio = (0.8, 0.1, 0.1) #80% imagenes entrenadas, 10% testing, 10% validacion


splitfolders.ratio(
    input_folder,
    output= output_folder,
    seed = 500, #para que cada ves que de dividan las imgs se haga de la misma forma
    ratio = split_ratio,
    group_prefix = None
)


In [14]:
img_size = (130, 162)
batch_size = 30

train_datagen = ImageDataGenerator(
    preprocessing_function= preprocess_input, #usa la funcion preproces input de resNet50
    rotation_range=20, #rota las imagenes aleatoreamente
    width_shift_range=0.2, # mueve la imagen de izquierda a la derecha aleatoriamente
    height_shift_range=0.2, #mueve la imagen arriba o abajo aleatoriamente
    shear_range=0.2,
    zoom_range= 0.2,
    horizontal_flip=True,
    fill_mode= 'nearest'
)

In [15]:
#ambas reescalan las imagenes para que los datos se mantengan
#aumento de datos para datos de testeo
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

#aumento de datos para datos de validacion
valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

In [17]:

train_dir = os.path.join(output_folder, "train")
val_dir = os.path.join(output_folder, "val")
test_dir = os.path.join(output_folder, "test")

#lee los datos de las carpetas y las prepara en tandas para entrenar el modelo
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=img_size, 
    batch_size=batch_size,
    class_mode='categorical'
)

valid_data = valid_datagen.flow_from_directory(
    val_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

test_data = test_datagen.flow_from_directory(
    test_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical'
)

Found 120 images belonging to 5 classes.
Found 15 images belonging to 5 classes.
Found 15 images belonging to 5 classes.


In [ ]:
'''import random
images, labels = next(valid_data)
idx = random.randint(0,images.shape[0]-1)

plt.imshow(images[idx])
plt.show()'''

In [26]:
from keras.applications.resnet import ResNet50 #es una cnn que esta entrenada en distintas categorias
os.environ['CUDA_VISIBLE_DEVICES'] = "0"
base_model = ResNet50(weights='imagenet',include_top=False,input_shape=(img_size[0], img_size[1], 3))
#top son clasificaciones, sin el podemos poner las que se necesiten

#razones :/
base_model.trainable = False

In [25]:
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    #ambas son fully connected layers
    layers.Dense(128, activation='relu'), #128 neuronas, con funcion de activacion relu
    layers.Dropout(0.5), # desactiva de forma aleatorea 50% de las variables de entrada
    layers.Dense(30, activation='softmax') #30 neuronas, funcion de activacion softmax
])


Compilacion del modelo

In [27]:
model.compile(optimizer='adam', #adaptive moment estimation, es eficiente
    loss='categorical_crossentropy',
    metrics=['accuracy'])

Entrenamiento del modelo

In [29]:
import scipy
model.fit(train_data, epochs=100, validation_data=valid_data)

Epoch 1/100


I0000 00:00:1778527455.882338  128340 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


ValueError: Arguments `target` and `output` must have the same shape. Received: target.shape=(None, 5), output.shape=(None, 30)